In [0]:
dbutils.fs.cp(
    "/Volumes/nyc_data/default/nyc",
    "/Volumes/nyc_data/bonze/nyc_data_brutes",
    recurse=True
)

In [0]:
dbutils.fs.rm(
    "dbfs:/Volumes/nyc_data/default/nyc",
    recurse=True
)

In [0]:
import os
catalog = "nyc_data"
schema = "bonze"
volume = "nyc_data_brutes"
file_name = "yellow_tripdata_2024-12.parquet"
table_name = "yellow_tripdata"
path_volume = "/Volumes/" + catalog + "/" + schema + "/" + volume
path_table = catalog + "." + schema

df = spark.read.parquet(
    os.path.join(path_volume, file_name)
)

### f"/Volumes/nyc_data/bonze/nyc_data_brutes/yellow_tripdata_2024-12.parquet"
#df.write.mode("overwrite").saveAsTable(f"{path_table}" + "." + f"{table_name}")

In [0]:
display(
    df.limit(5)
)

### Sauvegarde de la couche Bronze

In [0]:
(
    df.write
      .format("delta")
      .mode("overwrite")
      .saveAsTable("bronze_nyc_taxi")
)

In [0]:
%sql
select * from bronze_nyc_taxi
limit 10;

In [0]:
#df.printSchema()
#df.count()

In [0]:
#df = df.withColumnRenamed("tpep_pickup_datetime", "tpep_pickup_datetime2")
#df.printSchema()

In [0]:
print(f"Nombre de lignes : {df.count():,}")

In [0]:
display(df.describe())

In [0]:

from pyspark.sql.functions import col, count, when
null_counts = [
    (c, df.filter(col(c).isNull()).count())
    for c in df.columns
]
null_counts_df = spark.createDataFrame(null_counts, ["column", "null_count"])
display(null_counts_df)


## couche Silver

In [0]:
from pyspark.sql.functions import unix_timestamp
from pyspark.sql.functions import year, month, dayofmonth, col

df_silver = (
    df
    .withColumn(
        "trip_duration_min",
        (
        unix_timestamp("tpep_dropoff_datetime") 
        - unix_timestamp("tpep_pickup_datetime")
        ) / 60
    )
)

In [0]:
display(
    df_silver
    .limit(5)
)

In [0]:
df_silver = df_silver.filter(
    (df_silver.trip_duration_min > 0) &
    (df_silver.trip_distance > 0) &
    (df_silver.total_amount > 0)
)

In [0]:
from pyspark.sql.functions import col

df_silver = df_silver.filter(
    (col("trip_duration_min") > 0) &
    (col("trip_distance") > 0) &
    (col("total_amount") > 0)
)

In [0]:
query = """
    trip_duration_min > 0
    AND trip_distance > 0
    AND total_amount > 0
"""
df_silver = df_silver.filter(query)

In [0]:
from pyspark.sql.functions import hour, dayofweek, month, year

df_silver = (
    df_silver
    .withColumn(
        "pickup_hour",
        hour("tpep_pickup_datetime")
    )
    .withColumn(
        "pickup_day_of_week",
        dayofweek("tpep_pickup_datetime")
    )
    .withColumn(
        "pickup_month",
        month("tpep_pickup_datetime")
    )
    .withColumn(
        "pickup_year",
        year("tpep_pickup_datetime")
    )
)

In [0]:
display(
    df_silver
    .select("pickup_hour", "pickup_day_of_week", "pickup_month", "pickup_year")
    .limit(5)
)

## controle de qualité

In [0]:
bronze_count = df.count()
silver_count = df_silver.count()

print(f"Bronze : {bronze_count:,}")
print(f"Silver : {silver_count:,}")
print(f"Supprimées : {bronze_count-silver_count:,}")

### Sauvegarde Silver


In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_nyc_taxi")
)

### Construire la couche Gold

In [0]:
from pyspark.sql.functions import (
    col,
    year,
    month,
    dayofmonth,
    quarter,
    dayofweek,
    date_format
)

dim_date = (
    df_silver
    .select(col("tpep_pickup_datetime").cast("date").alias("date"))
    .distinct()
    .withColumn("year", year("date"))
    .withColumn("quarter", quarter("date"))
    .withColumn("month", month("date"))
    .withColumn("day", dayofmonth("date"))
    .withColumn("day_of_week", dayofweek("date"))
    .withColumn("month_name", date_format("date", "MMMM"))
    .withColumn("date_key",year(col("date")) * 10000 + month(col("date")) * 100 + dayofmonth(col("date")))
)

display(dim_date
        .limit(5)
)

In [0]:
display(
    dim_date.limit(5)
)

### Sauvegarde :

In [0]:
spark.sql("DROP TABLE IF EXISTS gold_dim_date")
(
    dim_date.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_dim_date")
)

### Créer la Fact Table

In [0]:
fact_trips = (
    df_silver.select(
        "VendorID",
        "PULocationID",
        "DOLocationID",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_duration_min",
        "trip_distance",
        "passenger_count",
        "fare_amount",
        "tip_amount",
        "total_amount",
        "payment_type"
    )
)

In [0]:
display(
    fact_trips.limit(5)
)

## Améllioration de gold_fact_trips

In [0]:
from pyspark.sql.functions import to_date

gold_fact_trips = (
    df_silver
    .withColumn(
        "trip_date",
        to_date("tpep_pickup_datetime")
    )
    .select(
        "trip_date",
        "PULocationID",
        "DOLocationID",
        "trip_duration_min",
        "trip_distance",
        "passenger_count",
        "fare_amount",
        "tip_amount",
        "total_amount",
        "payment_type"
    )
)

In [0]:
spark.sql("DROP TABLE IF EXISTS gold_fact_trips")
(
    gold_fact_trips.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_fact_trips")
)

In [0]:
display(
    df_silver.limit(5)
)

In [0]:
display(
    df_silver.limit(5).select(col("tpep_pickup_datetime"))
)

### Construire un KPI journalier

In [0]:
from pyspark.sql.functions import (
    sum,
    avg,
    count
)

kpi_daily = (
    df_silver
    .groupBy(
        col("tpep_pickup_datetime")
        .cast("date")
        .alias("trip_date")
    )
    .agg(
        count("*").alias("nb_trips"),
        sum("total_amount").alias("revenue"),
        avg("trip_distance").alias("avg_distance"),
        avg("tip_amount").alias("avg_tip")
    )
)

In [0]:
display(
    kpi_daily.limit(5)
)

In [0]:
(
    kpi_daily.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_kpi_daily")
)

### Ajouter les zones de taxi (très important)

In [0]:
df_location = spark.read.csv("/Volumes/nyc_data/ref_location/data_location/taxi_zone_lookup.csv", header=True)
display(
    df_location.limit(5)
)


In [0]:
(
    df_location.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("taxi_zone_lookup")
)

In [0]:
df_location.printSchema()

In [0]:
from pyspark.sql.functions import col

gold_dim_location = (
    df_location
    .withColumn(
        "location_id",
        col("LocationID").cast("int")
    )
    .select(
        "location_id",
        "Borough",
        "Zone",
        "service_zone"
    )
)

In [0]:
gold_dim_location.printSchema()

display(gold_dim_location)

In [0]:
(
    gold_dim_location.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_dim_location")
)

In [0]:
%sql
select * 
from gold_dim_location
;

In [0]:
%sql
select * 
from taxi_zone_lookup
where LocationID = 161
;

### Vérifier les tables créées

In [0]:
%sql
SHOW TABLES;

#### Creation de vue métier

In [0]:
spark.table("gold_dim_date").printSchema()

In [0]:
%sql
DESCRIBE gold_dim_date;

#### Top 10 zones de départ

In [0]:
%sql
select Borough, count(*) as Count
from silver_nyc_taxi as m
inner join taxi_zone_lookup as z on m.PULocationID = z.LocationID
group by Borough
order by count(*) desc
limit 11
;

In [0]:
%sql
SELECT
    Zone,
    COUNT(*) AS nb_trips
FROM silver_nyc_taxi m
JOIN taxi_zone_lookup z
    ON m.DOLocationID = z.LocationID
GROUP BY Zone
ORDER BY nb_trips DESC
LIMIT 10

In [0]:
%sql
SELECT
    Borough,
    ROUND(SUM(total_amount), 2) AS revenue
FROM silver_nyc_taxi m
JOIN taxi_zone_lookup z
    ON m.PULocationID = z.LocationID
GROUP BY Borough
ORDER BY revenue DESC

In [0]:
(
    fact_trips
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv("/tmp/fact_trips")
)

### Alimentation vers SQL Server local

In [0]:
spark.sql("SHOW VOLUMES").display()
spark.sql("SHOW CATALOGS").display()
spark.sql("SHOW SCHEMAS IN <catalog>").display()

In [0]:
from pyspark.sql.functions import format_number

df = spark.table("gold_fact_trips")
df_export = (
    df
    .withColumn("fare_amount", format_number("fare_amount", 2))
    .withColumn("tip_amount", format_number("tip_amount", 2))
    .withColumn("total_amount", format_number("total_amount", 2))
)

tmp_dir = "/Volumes/nyc_data/gold/data_bi/_tmp_export"

(df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(tmp_dir)
)

# récupérer le part-xxxx.csv
part_file = [
    f.path
    for f in dbutils.fs.ls(tmp_dir)
    if f.name.startswith("part-") and f.name.endswith(".csv")
][0]

# nom final souhaité
final_file = "/Volumes/nyc_data/gold/data_bi/gold_fact_trips.csv"

# déplacer + renommer
dbutils.fs.mv(part_file, final_file)

# nettoyage
dbutils.fs.rm(tmp_dir, recurse=True)

print(final_file)


In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
SHOW SCHEMAS IN nyc_data;

In [0]:
spark.sql("SHOW SCHEMAS IN nyc_data").show()

In [0]:
%sql
SHOW VOLUMES IN nyc_data.default;

In [0]:
spark.sql("SHOW VOLUMES IN nyc_data.default").show(truncate=False)

### alimentation vers Azure SQL en entreprise

In [0]:
jdbc_url = (
    "jdbc:sqlserver://myserver.database.windows.net:1433;"
    "database=DW_NYC_TAXI;"
    "encrypt=true;"
    "trustServerCertificate=false;"
)

In [0]:
(gold_fact_trips.write
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "fact_trips")
    .option("user", username)
    .option("password", password)
    .mode("append")
    .save())

In [0]:
import os

os.listdir(
    "/Workspace/Users/alphadiop@gmail.com/Learning workspace/src/nyc"
)